In [1]:
# ===============================
# TASK 15 — SPARK DATAFRAME OPS
# ===============================

from pyspark.sql import SparkSession
from pyspark.sql.functions import col, when, avg, sum

print("\n===== TASK 15 STARTED =====")

# Create Spark Session
spark = SparkSession.builder \
    .appName("Spark DataFrame Operations") \
    .master("local[1]") \
    .getOrCreate()

# Load Dataset
df = spark.read.csv(
    "../data/faculty_workload_raw.csv",
    header=True,
    inferSchema=True
)

print("\nOriginal Data:")
df.show()

# ===============================
# CLEANING STEP
# ===============================

print("\nCleaning Data...")

# Fix negative hours → convert to positive
df = df.withColumn(
    "hours_assigned",
    when(col("hours_assigned") < 0,
         col("hours_assigned") * -1)
    .otherwise(col("hours_assigned"))
)

# Replace null values → 0
df = df.fillna({
    "hours_assigned": 0
})

print("\nCleaned Data:")
df.show()

# ===============================
# FILTERING STEP
# ===============================

print("\nFiltering Data (hours > 2)...")

filtered_df = df.filter(
    col("hours_assigned") > 2
)

filtered_df.show()

# ===============================
# AGGREGATION STEP
# ===============================

print("\nDepartment Workload Summary:")

dept_summary = df.groupBy("department") \
    .agg(
        sum("hours_assigned").alias("total_hours"),
        avg("hours_assigned").alias("avg_hours")
    )

dept_summary.show()

# ===============================
# SAVE OUTPUT
# ===============================

pandas_df = dept_summary.toPandas()

pandas_df.to_csv(
    "../data/department_summary.csv",
    index=False
)

print("\nOutput Saved Successfully")

spark.stop()

print("\n===== TASK 15 COMPLETED =====")


===== TASK 15 STARTED =====

Original Data:
+----------+------------+----------+--------------+--------------+--------+
|faculty_id|faculty_name|department|       subject|hours_assigned|semester|
+----------+------------+----------+--------------+--------------+--------+
|         1| Amit Sharma|       CSE|  Data Science|             4|    Sem1|
|         2|    Riya Das|       ECE|      Circuits|             3|    Sem1|
|         3|   John Paul|        IT|          DBMS|             5|    Sem2|
|         4|   Sara Khan|        ME|Thermodynamics|             2|    Sem2|
|         5|   Raj Patel|       CSE|            AI|            -4|    Sem1|
|         6|  Neha Singh|       ECE|       Signals|          null|    Sem1|
|         7|  Arun Kumar|        IT|      Networks|             5|    Sem2|
|         8|   Meena Roy|        ME|      Robotics|             2|    Sem2|
+----------+------------+----------+--------------+--------------+--------+


Cleaning Data...

Cleaned Data:
+--------